<!-- Здесь проверяем, насколько различаются ответы модели на один запрос при выборке как в роллаутах RL. -->
# Разнообразие ответов при T = 1

GRPO учится на различиях внутри группы ответов на один запрос. Если модель после своего дообучения
стала слишком «острой» (почти всегда один и тот же ответ), награды в группе одинаковые и учиться не на чем.
Здесь проверяем это до обучения, на двух кандидатах: T-lite-it-2.1 и Qwen3-4B-Instruct-2507.

**Что сравниваем.** По 8 ответов на каждую тему в двух режимах выборки:

| Режим | Настройки | Зачем |
|---|---|---|
| `t1` | T = 1, без top-k, top-p и штрафов | ровно распределение модели — так будут идти роллауты |
| `t1_presence` | то же + `presence_penalty = 1.0` | штраф, который T-lite требует против повторов |

`presence_penalty` в transformers нет, он реализован ниже вручную (`PresencePenalty`): из логита каждого
токена, который уже встречался в ответе, вычитается 1.0. Штраф — детерминированная функция уже
написанного текста, поэтому его можно считать частью политики, если при подсчёте log-prob для PPO/GRPO
применять ту же маску.

**Что меряем для каждого ответа:** число токенов, обрыв по лимиту, средняя энтропия выбора токена,
доля повторяющихся 4-грамм (зацикливание), заглушки вроде «[Ваше имя]».
**Для группы из 8:** насколько ответы различаются между собой (1 − среднее сходство по 3-граммам слов).

**Куда пишутся ответы.** Каталог `sampling_diversity/` (в Kaggle — в `/kaggle/working`), три файла:
- `results.jsonl` — сырые данные: все ответы и замеры, по строке на ответ; дописывается после каждой темы,
  поэтому при падении готовое не теряется;
- `summary.csv` — сводная таблица;
- `compare.html` — для чтения в браузере: по каждой теме таблица, столбцы — режимы, строки — номер ответа.

После скачивания с Kaggle файлы кладём в `runs/sampling_diversity/` проекта (папка `runs/` в git не попадает).

**Kaggle.** Accelerator — GPU T4, Internet — On. Save Version → Save & Run All.
На T4 — около 2 часов, если ответы зацикливаются до лимита — до 3. Ноутбук самостоятельный, код проекта не нужен.

Обработчики `PresencePenalty` и `EntropyMeter` здесь временные: при реализации роллаутов
они переезжают в `src/presentation_rl/training/`.

## Установка

В Kaggle нет bitsandbytes (нужен для T-lite в 4 битах): ячейка ниже ставит его, если его нет.
Ставить нужно **до первого импорта transformers** — transformers проверяет наличие bitsandbytes при импорте,
поэтому после установки в уже работающей сессии нужен перезапуск.
Ошибка про версию `tokenizers` при импорте transformers лечится так: `!pip install -q "tokenizers>=0.22.0,<=0.23.0"`
(границы — из текста ошибки), затем перезапуск сессии.

In [1]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("bitsandbytes") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.46.1"], check=True)
    print("bitsandbytes установлен")
for name in ("torch", "transformers", "accelerate", "bitsandbytes", "pandas"):
    print(f"{name:14s}", "есть" if importlib.util.find_spec(name) else "НЕТ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.3 MB/s eta 0:00:00
bitsandbytes установлен
torch          есть
transformers   есть
accelerate     есть
bitsandbytes   есть
pandas         есть


## Импорты

Все импорты — здесь, после установки bitsandbytes: transformers проверяет его наличие при импорте.

In [2]:
import gc, html, itertools, json, re, time
from pathlib import Path

import pandas as pd
import torch
import transformers
from packaging.version import Version
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList,
)

## Настройки

In [3]:
MODELS = {
    # 8B: на T4 только в 4 битах (≈6,5 ГБ). Пачка 8 на T4 не помещается: кэш длинных ответов растёт к концу
    # генерации, и нехватка памяти наступает почти на финише. С пачкой 4 пик ≈ 12,5 ГБ из 14,6.
    "tlite_2.1": {"model_id": "t-tech/T-lite-it-2.1", "load_in_4bit": True, "batch": 4, "chat_template_kwargs": {}},
    # 4B в fp16 ≈ 8 ГБ, поэтому пачка поменьше.
    "qwen3_4b": {"model_id": "Qwen/Qwen3-4B-Instruct-2507", "load_in_4bit": False, "batch": 4, "chat_template_kwargs": {}},
}

# Режимы выборки. top_k=0, top_p=1.0, repetition_penalty=1.0 явно отключают настройки
# из generation_config модели (там стоят T=0.7, top_p=0.8, top_k=20).
SAMPLING = {
    "t1": {"temperature": 1.0, "top_k": 0, "top_p": 1.0, "repetition_penalty": 1.0, "presence_penalty": 0.0},
    "t1_presence": {"temperature": 1.0, "top_k": 0, "top_p": 1.0, "repetition_penalty": 1.0, "presence_penalty": 1.0},
}

# Что запускаем: модель → режимы.
RUNS = {
    "tlite_2.1": ["t1", "t1_presence"],
    "qwen3_4b": ["t1", "t1_presence"],
}

N_SAMPLES = 8          # ответов на тему в одном режиме — как размер группы GRPO
SLIDES_MIN, SLIDES_MAX = 10, 12   # диапазон слайдов из задания; попадание в него меряется
MAX_NEW_TOKENS = 4096
SEED = 42

IN_KAGGLE = Path("/kaggle/working").exists()
OUT_DIR = Path("/kaggle/working/sampling_diversity") if IN_KAGGLE else Path("../runs/sampling_diversity")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Ответы пишутся в", OUT_DIR.resolve())

Ответы пишутся в /kaggle/working/sampling_diversity


## Задание

Аудитории — студенты или специалисты в своей области. Требования к форме записаны явно:
без заглушек, без вступлений и концовок вокруг презентации, без выдуманных источников.

In [4]:
TOPICS = [
    {"id": "gradient_descent", "topic": "Градиентный спуск и выбор шага обучения",
     "audience": "студенты второго курса технического вуза, знают производную и основы линейной алгебры"},
    {"id": "photosynthesis", "topic": "Фотосинтез: как растения превращают свет в химическую энергию",
     "audience": "студенты первого курса биологического факультета"},
    {"id": "serf_reform", "topic": "Отмена крепостного права в России",
     "audience": "учителя истории на курсах повышения квалификации"},
]

SYSTEM_PROMPT = "Ты — опытный преподаватель и автор учебных презентаций. Отвечай на русском языке."

PROMPT_TEMPLATE = (
    "Подготовь содержание презентации на тему «{topic}».\n"
    "Аудитория: {audience}.\n"
    "Презентацию покажут на занятии, поэтому текст должен быть готов к показу без доработки.\n\n"
    "Требования:\n"
    "1. {slides_min}–{slides_max} слайдов. Для каждого слайда: заголовок; текст слайда связными предложениями, 60–100 слов; "
    "какое изображение, график или схему поставить и что именно на них показать; формулы в LaTeX — только если уместны.\n"
    "2. Раскрой тему глубоко: механизмы и причины, конкретные примеры, ограничения и спорные вопросы.\n"
    "3. Пиши только то, в чём уверен. Не придумывай источники, цитаты, даты и числа.\n"
    "4. Никаких заглушек и шаблонных полей: не пиши «[Ваше имя]», «[дата]», «[название вуза]», "
    "не добавляй слайды «Спасибо за внимание» и «Вопросы».\n"
    "5. Выведи только слайды: без вступления перед ними и без комментариев после."
)


def build_messages(task):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": PROMPT_TEMPLATE.format(**task, slides_min=SLIDES_MIN, slides_max=SLIDES_MAX)},
    ]


print(build_messages(TOPICS[0])[1]["content"])

Подготовь содержание презентации на тему «Градиентный спуск и выбор шага обучения».
Аудитория: студенты второго курса технического вуза, знают производную и основы линейной алгебры.
Презентацию покажут на занятии, поэтому текст должен быть готов к показу без доработки.

Требования:
1. 10–12 слайдов. Для каждого слайда: заголовок; текст слайда связными предложениями, 60–100 слов; какое изображение, график или схему поставить и что именно на них показать; формулы в LaTeX — только если уместны.
2. Раскрой тему глубоко: механизмы и причины, конкретные примеры, ограничения и спорные вопросы.
3. Пиши только то, в чём уверен. Не придумывай источники, цитаты, даты и числа.
4. Никаких заглушек и шаблонных полей: не пиши «[Ваше имя]», «[дата]», «[название вуза]», не добавляй слайды «Спасибо за внимание» и «Вопросы».
5. Выведи только слайды: без вступления перед ними и без комментариев после.


## Устройство

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8:
    DTYPE = torch.bfloat16   # настоящий bf16 — только с Ampere; T4 его лишь эмулирует
elif DEVICE == "cuda":
    DTYPE = torch.float16
else:
    DTYPE = torch.float32

if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} ГБ")
    print("тип чисел:", DTYPE)
else:
    print("Видеокарта не найдена — на процессоре это займёт сутки")


def reset_peak_memory():
    if DEVICE == "cuda":
        for i in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(i)


def peak_memory_gb():
    # Сумма по всем GPU: при device_map="auto" модель может лежать на нескольких картах.
    if DEVICE != "cuda":
        return None
    total = sum(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count()))
    return round(total / 1024**3, 2)

GPU 0: Tesla T4, 14.6 ГБ
GPU 1: Tesla T4, 14.6 ГБ
тип чисел: torch.float16


## Штраф за присутствие и замер энтропии

Оба — обработчики логитов (`LogitsProcessor`): transformers вызывает их на каждом шаге генерации,
до выбора токена.

- `PresencePenalty` вычитает `penalty` из логита каждого токена, который уже есть в ответе
  (только в ответе, не в промпте — как в vLLM и API OpenAI).
- `EntropyMeter` ничего не меняет, только считает энтропию распределения, из которого выбирается токен:
  H = −Σ p·log p, в натах. Около 0 — модель почти уверена в одном токене; чем больше, тем шире выбор.
  Шаги после конца ответа не учитываются. Ставится после штрафа, чтобы мерить распределение уже со штрафом.
  При T = 1 без top-k и top-p других преобразований логитов после него нет — это ровно то распределение, из которого выбирается токен.

In [6]:
class PresencePenalty(LogitsProcessor):
    def __init__(self, penalty, prompt_length):
        self.penalty = penalty
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        generated = input_ids[:, self.prompt_length:]
        if self.penalty == 0 or generated.shape[1] == 0:
            return scores
        seen = torch.zeros_like(scores, dtype=torch.bool)
        seen.scatter_(1, generated, True)
        return scores - self.penalty * seen.to(scores.dtype)


class EntropyMeter(LogitsProcessor):
    def __init__(self, eos_ids, prompt_length):
        self.eos_ids = torch.tensor(sorted(eos_ids))
        self.prompt_length = prompt_length
        self.total = None   # сумма энтропий по шагам, на каждый ответ
        self.steps = None   # число учтённых шагов, на каждый ответ
        self.done = None    # ответ уже закончился

    def __call__(self, input_ids, scores):
        if self.total is None:
            batch = scores.shape[0]
            self.total = torch.zeros(batch, device=scores.device)
            self.steps = torch.zeros(batch, device=scores.device)
            self.done = torch.zeros(batch, dtype=torch.bool, device=scores.device)
        if input_ids.shape[1] > self.prompt_length:
            last = input_ids[:, -1]
            self.done |= torch.isin(last, self.eos_ids.to(last.device))
        log_p = torch.log_softmax(scores.float(), dim=-1)
        entropy = -(log_p.exp() * log_p).nan_to_num().sum(dim=-1)
        active = (~self.done).float()
        self.total += entropy * active
        self.steps += active
        return scores

    def mean(self):
        return (self.total / self.steps.clamp(min=1)).tolist()

## Метрики текста

- `rep4` — доля повторяющихся 4-грамм слов внутри ответа: 0 — повторов нет; > 0,3 — похоже на зацикливание.
- `placeholders` — найденные заглушки.
- `slides` — число слайдов: различные номера в строках, начинающихся со «Слайд N»
  (перед словом допускаются `#`, `*`, эмодзи). Если таких строк нет — нумерованные заголовки
  Markdown вида `### 1. Введение`. Эвристика для свободного текста; на прогоне EXP-002 совпала с ручным подсчётом во всех 15 ответах.
- `in_range` — число слайдов в диапазоне из задания.
- `diversity` группы — 1 − среднее попарное сходство Жаккара множеств 3-грамм слов по всем парам из 8 ответов:
  0 — ответы одинаковые, 1 — ни одной общей тройки слов.

In [7]:
PLACEHOLDER = re.compile(
    r"\[(?:ваш|ваше|имя|фио|дата|число|название|автор|преподаватель|лектор|школ|вуз|университет|курс|ссылк|логотип)[^\]]{0,40}\]"
    r"|спасибо за внимание",
    re.IGNORECASE,
)


def words(text):
    return re.findall(r"\w+", text.lower())


def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def rep4(text):
    grams = ngrams(words(text), 4)
    return round(1 - len(set(grams)) / len(grams), 3) if grams else 0.0


def placeholders(text):
    return sorted({m.group(0) for m in PLACEHOLDER.finditer(text)})


SLIDE_LINE = re.compile(r"(?im)^\W*(?:слайд|slide)\s*№?\s*(\d+)")
NUMBERED_HEADING = re.compile(r"(?m)^#{1,6}\s*\**\s*(\d+)\s*[.:)]")


def count_slides(text):
    numbers = SLIDE_LINE.findall(text) or NUMBERED_HEADING.findall(text)
    return len(set(numbers))


def diversity(texts):
    sets = [set(ngrams(words(t), 3)) for t in texts]
    pairs = [(a, b) for a, b in itertools.combinations(sets, 2) if a or b]
    if not pairs:
        return None
    similarity = sum(len(a & b) / len(a | b) for a, b in pairs) / len(pairs)
    return round(1 - similarity, 3)


print(rep4("один два три четыре один два три четыре один два три четыре"), placeholders("Автор: [Ваше имя], [дата]"))
print(count_slides("### **Слайд 1: Введение**\nтекст\n## 🔹 Слайд 2. Суть\n"), count_slides("### **1. Введение**\n### **2. Итог**"))

0.556 ['[Ваше имя]', '[дата]']
2 2


## Загрузка и генерация

`run_model` загружает модель, для каждого её режима и каждой темы генерирует 8 ответов,
дописывает их в `results.jsonl` и выгружает модель.
- Повторный запуск продолжает с места остановки: готовые группы (режим + тема) из `results.jsonl` пропускаются.
  `run_model(name, fresh=True)` удаляет прежние ответы модели и начинает заново.
- Ответы генерируются пачками (`batch` в `MODELS`). Если памяти не хватило, пачка делится пополам,
  и уменьшенный размер запоминается для следующих групп — неудачная попытка не повторяется на каждой теме.

In [8]:
DTYPE_KEY = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"


def load_model(name):
    cfg = MODELS[name]
    kwargs = {DTYPE_KEY: DTYPE}
    if cfg["load_in_4bit"] and DEVICE == "cuda":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE
        )
        kwargs["device_map"] = {"": 0}   # целиком на первую карту: 8B в nf4 на T4 помещается
    started = time.time()
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"])
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], **kwargs)
    if "device_map" not in kwargs:
        model = model.to(DEVICE)
    model.eval()
    print(f"{name} → {cfg['model_id']} | загрузка {time.time() - started:.0f} с | память {peak_memory_gb()} ГБ")
    return model, tokenizer


def eos_ids(model, tokenizer):
    ids = model.generation_config.eos_token_id
    ids = set(ids if isinstance(ids, list) else [ids])
    if tokenizer.eos_token_id is not None:
        ids.add(tokenizer.eos_token_id)
    return ids


def generate_batch(model, tokenizer, inputs, sampling, n, eos):
    prompt_length = inputs["input_ids"].shape[1]
    meter = EntropyMeter(eos, prompt_length)
    processors = LogitsProcessorList([PresencePenalty(sampling["presence_penalty"], prompt_length), meter])
    params = {k: v for k, v in sampling.items() if k != "presence_penalty"}
    with torch.no_grad():
        output = model.generate(
            **inputs, do_sample=True, num_return_sequences=n, max_new_tokens=MAX_NEW_TOKENS,
            logits_processor=processors, pad_token_id=tokenizer.pad_token_id or min(eos), **params,
        )
    results = []
    for row, entropy in zip(output[:, prompt_length:].tolist(), meter.mean()):
        end = next((i for i, token in enumerate(row) if token in eos), None)
        length = end + 1 if end is not None else len(row)
        results.append({
            "text": tokenizer.decode(row[:length], skip_special_tokens=True),
            "output_tokens": length,
            "truncated": end is None,
            "mean_entropy": round(entropy, 3),
        })
    return results


def generate_group(model, tokenizer, name, mode, task, seed):
    cfg, sampling = MODELS[name], SAMPLING[mode]
    inputs = tokenizer.apply_chat_template(
        build_messages(task), add_generation_prompt=True, return_tensors="pt", return_dict=True,
        **cfg["chat_template_kwargs"],
    ).to(model.device)
    eos = eos_ids(model, tokenizer)
    torch.manual_seed(seed)
    reset_peak_memory()
    started = time.time()
    results, batch = [], cfg["batch"]
    while len(results) < N_SAMPLES:
        n = min(batch, N_SAMPLES - len(results))
        try:
            results += generate_batch(model, tokenizer, inputs, sampling, n, eos)
        except torch.cuda.OutOfMemoryError:
            if batch == 1:
                raise
            batch = max(1, batch // 2)
            cfg["batch"] = batch   # следующие группы сразу начинают с уменьшенной пачки
            gc.collect()
            torch.cuda.empty_cache()
            print(f"  не хватило памяти, пачка уменьшена до {batch}")
    seconds = time.time() - started
    for i, r in enumerate(results):
        r.update({
            "model": name, "model_id": cfg["model_id"], "mode": mode, "sampling": sampling,
            "task": task["id"], "sample": i, "seed": seed, "batch": batch,
            "rep4": rep4(r["text"]), "placeholders": placeholders(r["text"]),
            "slides": count_slides(r["text"]),
        })
        r["in_range"] = SLIDES_MIN <= r["slides"] <= SLIDES_MAX
    tokens = sum(r["output_tokens"] for r in results)
    print(
        f"  {task['id']:17s} {seconds / 60:5.1f} мин | {tokens / seconds:5.1f} ток/с | память {peak_memory_gb()} ГБ | "
        f"токенов {tokens // N_SAMPLES} в среднем | обрывов {sum(r['truncated'] for r in results)} | "
        f"энтропия {sum(r['mean_entropy'] for r in results) / N_SAMPLES:.2f} | "
        f"слайдов в диапазоне {sum(r['in_range'] for r in results)}/{N_SAMPLES} | "
        f"разнообразие {diversity([r['text'] for r in results])}"
    )
    return results


RESULTS = OUT_DIR / "results.jsonl"


def drop_old_records(name):
    # Удаляет прежние ответы модели, ответы других моделей остаются.
    if not RESULTS.exists():
        return
    keep = [line for line in RESULTS.open(encoding="utf-8") if json.loads(line)["model"] != name]
    RESULTS.write_text("".join(keep), encoding="utf-8")


def done_groups(name):
    # Группы (режим, тема), для которых в файле уже есть все N_SAMPLES ответов.
    if not RESULTS.exists():
        return set()
    counts = {}
    for line in RESULTS.open(encoding="utf-8"):
        r = json.loads(line)
        if r["model"] == name:
            counts[(r["mode"], r["task"])] = counts.get((r["mode"], r["task"]), 0) + 1
    return {key for key, n in counts.items() if n >= N_SAMPLES}


def run_model(name, fresh=False):
    model = tokenizer = None
    try:
        if fresh:
            drop_old_records(name)
        done = done_groups(name)
        todo = [(mode, task) for mode in RUNS[name] for task in TOPICS if (mode, task["id"]) not in done]
        if not todo:
            print(f"{name}: все группы уже есть в {RESULTS.name}")
            return
        if done:
            print(f"{name}: готово групп {len(done)}, осталось {len(todo)}")
        model, tokenizer = load_model(name)
        for m, mode in enumerate(RUNS[name]):
            print(f"\n{name} | {mode}")
            for t, task in enumerate(TOPICS):
                if (mode, task["id"]) in done:
                    print(f"  {task['id']:17s} уже есть, пропуск")
                    continue
                group = generate_group(model, tokenizer, name, mode, task, SEED + 100 * t + m)
                with RESULTS.open("a", encoding="utf-8") as f:
                    f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in group)
    except Exception as exc:   # упавшая модель не останавливает следующую
        print(f"\n!!! {name}: {type(exc).__name__}: {exc}")
    finally:
        del model, tokenizer
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        print(f"{name}: память освобождена")

## T-lite-it-2.1

In [9]:
run_model("tlite_2.1")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tlite_2.1 → t-tech/T-lite-it-2.1 | загрузка 122 с | память 7.37 ГБ

tlite_2.1 | t1
  gradient_descent   28.6 мин |  11.6 ток/с | память 8.8 ГБ | токенов 2489 в среднем | обрывов 0 | энтропия 1.13 | слайдов в диапазоне 8/8 | разнообразие 0.986
  photosynthesis     19.0 мин |  15.2 ток/с | память 8.22 ГБ | токенов 2170 в среднем | обрывов 0 | энтропия 1.58 | слайдов в диапазоне 7/8 | разнообразие 0.986
  serf_reform        29.8 мин |   8.9 ток/с | память 9.31 ГБ | токенов 1989 в среднем | обрывов 1 | энтропия 1.73 | слайдов в диапазоне 7/8 | разнообразие 0.991

tlite_2.1 | t1_presence
  gradient_descent   42.3 мин |  10.3 ток/с | память 9.33 ГБ | токенов 3268 в среднем | обрывов 3 | энтропия 3.48 | слайдов в диапазоне 6/8 | разнообразие 0.995
  photosynthesis     35.7 мин |  10.5 ток/с | память 9.32 ГБ | токенов 2822 в среднем | обрывов 2 | энтропия 3.35 | слайдов в диапазоне 4/8 | разнообразие 0.99
  serf_reform        35.9 мин |  10.5 ток/с | память 9.11 ГБ | токенов 2839 в среднем | о

## Qwen3-4B-Instruct-2507

In [10]:
run_model("qwen3_4b")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

qwen3_4b → Qwen/Qwen3-4B-Instruct-2507 | загрузка 53 с | память 9.11 ГБ

qwen3_4b | t1
  gradient_descent   12.3 мин |  22.2 ток/с | память 9.53 ГБ | токенов 2048 в среднем | обрывов 0 | энтропия 0.50 | слайдов в диапазоне 8/8 | разнообразие 0.974
  photosynthesis     14.0 мин |  22.1 ток/с | память 9.68 ГБ | токенов 2312 в среднем | обрывов 0 | энтропия 0.43 | слайдов в диапазоне 8/8 | разнообразие 0.963
  serf_reform        14.6 мин |  21.5 ток/с | память 9.74 ГБ | токенов 2359 в среднем | обрывов 0 | энтропия 0.83 | слайдов в диапазоне 8/8 | разнообразие 0.99

qwen3_4b | t1_presence
  gradient_descent   11.3 мин |  23.8 ток/с | память 9.47 ГБ | токенов 2011 в среднем | обрывов 0 | энтропия 0.55 | слайдов в диапазоне 8/8 | разнообразие 0.977
  photosynthesis     14.3 мин |  20.6 ток/с | память 9.75 ГБ | токенов 2212 в среднем | обрывов 0 | энтропия 0.48 | слайдов в диапазоне 8/8 | разнообразие 0.971
  serf_reform        14.9 мин |  20.4 ток/с | память 9.79 ГБ | токенов 2278 в среднем

## Сводка и файлы для чтения

Ячейки можно запускать отдельно, в любой момент: они читают `results.jsonl`.

In [11]:
records = [json.loads(line) for line in RESULTS.open(encoding="utf-8")]
runs = list(dict.fromkeys(f"{r['model']}__{r['mode']}" for r in records))
print(f"ответов: {len(records)}, режимов: {len(runs)}")

rows = []
for run in runs:
    for task in TOPICS:
        group = [r for r in records if f"{r['model']}__{r['mode']}" == run and r["task"] == task["id"]]
        if not group:
            continue
        rows.append({
            "run": run, "task": task["id"], "n": len(group),
            "tokens": round(sum(r["output_tokens"] for r in group) / len(group)),
            "truncated": sum(r["truncated"] for r in group),
            "entropy": round(sum(r["mean_entropy"] for r in group) / len(group), 3),
            "rep4": round(sum(r["rep4"] for r in group) / len(group), 3),
            "looping": sum(r["rep4"] > 0.3 for r in group),
            "with_placeholders": sum(bool(r["placeholders"]) for r in group),
            "slides": round(sum(r["slides"] for r in group) / len(group), 1),
            "in_range": sum(r["in_range"] for r in group),
            "diversity": diversity([r["text"] for r in group]),
        })
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary.csv", index=False)
display(summary)

ответов: 96, режимов: 4


,run,task,n,tokens,truncated,entropy,rep4,looping,with_placeholders,slides,in_range,diversity
0,tlite_2.1__t1,gradient_descent,8,2489,0,1.127,0.016,0,0,11.9,8,0.986
1,tlite_2.1__t1,photosynthesis,8,2171,0,1.583,0.000,0,0,10.2,7,0.986
2,tlite_2.1__t1,serf_reform,8,1989,1,1.731,0.029,0,0,9.2,7,0.991
3,tlite_2.1__t1_presence,gradient_descent,8,3268,3,3.478,0.007,0,0,9.6,6,0.995
4,tlite_2.1__t1_presence,photosynthesis,8,2822,2,3.346,0.004,0,0,8.2,4,0.990
5,tlite_2.1__t1_presence,serf_reform,8,2840,0,2.892,0.002,0,0,9.4,4,0.995
6,qwen3_4b__t1,gradient_descent,8,2048,0,0.496,0.006,0,0,11.0,8,0.974
7,qwen3_4b__t1,photosynthesis,8,2313,0,0.432,0.003,0,0,11.0,8,0.963
8,qwen3_4b__t1,serf_reform,8,2359,0,0.832,0.002,0,0,11.0,8,0.990
9,qwen3_4b__t1_presence,gradient_descent,8,2012,0,0.555,0.002,0,0,11.0,8,0.977


In [12]:
# Одна страница: по каждой теме таблица, столбцы — режимы, строки — номер ответа.
STYLE = (
    "body{font-family:sans-serif;margin:16px} table{border-collapse:collapse;table-layout:fixed;width:100%;margin-bottom:40px}"
    "td,th{border:1px solid #ccc;vertical-align:top;padding:6px} th{background:#eef}"
    "pre{white-space:pre-wrap;font:13px/1.45 sans-serif;margin:0} .meta{color:#666;font-size:12px;margin-bottom:4px}"
)
sections = []
for task in TOPICS:
    by_run = {run: {r["sample"]: r for r in records if f"{r['model']}__{r['mode']}" == run and r["task"] == task["id"]} for run in runs}
    head = "".join(
        f"<th>{html.escape(run)}<br><span class='meta'>разнообразие {diversity([r['text'] for r in by_run[run].values()])}</span></th>"
        for run in runs
    )
    body = []
    for i in range(N_SAMPLES):
        cells_html = []
        for run in runs:
            r = by_run[run].get(i)
            if r is None:
                cells_html.append("<td></td>")
                continue
            meta = f"слайдов {r['slides']} · токенов {r['output_tokens']} · энтропия {r['mean_entropy']} · rep4 {r['rep4']}"
            meta += " · <b>обрыв</b>" if r["truncated"] else ""
            meta += f" · заглушки: {html.escape(', '.join(r['placeholders']))}" if r["placeholders"] else ""
            cells_html.append(f"<td><div class='meta'>{meta}</div><pre>{html.escape(r['text'])}</pre></td>")
        body.append(f"<tr><th style='width:30px'>{i + 1}</th>{''.join(cells_html)}</tr>")
    sections.append(
        f"<h2 id='{task['id']}'>{html.escape(task['topic'])}</h2><p>Аудитория: {html.escape(task['audience'])}</p>"
        f"<table><tr><th style='width:30px'>№</th>{head}</tr>{''.join(body)}</table>"
    )
contents = " · ".join(f"<a href='#{t['id']}'>{html.escape(t['topic'])}</a>" for t in TOPICS)
page = (
    f"<!doctype html><meta charset='utf-8'><title>Разнообразие ответов</title><style>{STYLE}</style>"
    f"<p>{contents}</p>{''.join(sections)}"
)
(OUT_DIR / "compare.html").write_text(page, encoding="utf-8")
print("готово:", *sorted(p.name for p in OUT_DIR.iterdir()), sep="\n  ")

готово:
  compare.html
  results.jsonl
  summary.csv


## Как читать результат

| Показатель | Хорошо для GRPO | Плохо |
|---|---|---|
| `diversity` | заметно больше 0: разные планы, примеры, формулировки | около 0 — ответы почти одинаковые, группе не из чего выбирать |
| `entropy` | умеренная (примерно 0,5–2 ната) | около 0 — модель «острая»; очень высокая — выборка мусора |
| `rep4`, `looping` | низкие | ответы зацикливаются — без штрафа T = 1 непригодна |
| `truncated` | 0 | обрывы по лимиту обычно означают зацикливание |
| `with_placeholders` | 0 | модель не слушается требований к форме |
| `in_range` | 8 из 8 | модель не держит заданное число слайдов — на это нельзя опираться как на критерий |

Числа — только первый фильтр. Обязательно прочитать несколько ответов в `compare.html`:
высокое «разнообразие» может оказаться разнообразием ошибок.